In [ ]:
# ==== CONFIG (shared) ====
PROJECT          = "JAXUED_TEST"
LEARNING_RATES   = [6e-4] # [3e-5, 7.5e-5, 1.5e-4, 2e-4, 2.5e-4, 4.5e-4, 6e-4]
SEEDS            = [16] # [10, 11, 12, 13, 14, 15, 16]  # One seed per LR
NUM_UPDATES      = 30_000
EVAL_FREQ        = 250
EVAL_NUM         = 80               # Changed from 250 to 80
PERSIST_TO_DRIVE = True
DRIVE_DIR_NAME   = "jaxued_checkpoints"

# * CHECK YOUR COLAB CREDITS BEFORE RUNNING *

# Repo pinning (optional)
REPO_URL      = "https://github.com/callum-lawson/jaxued.git"
REPO_DIR      = "jaxued"
GIT_CHECKOUT  = None                 # e.g. "main" or "my-branch"
GIT_COMMIT    = None                 # e.g. "a1b2c3d"

# Define the run TYPES for each algorithm
RUNS = [
    {
        "RUN_NAME":  "dr_baseline_eval80_seed0_lr{lr}_30000c",   # {lr} will be replaced
        "SCRIPT":    "examples/maze_dr.py",
        "USE_ACCEL": False,
    },
    {
        "RUN_NAME":  "dr_softmin_eval80_seed0_lr{lr}_30000c",
        "SCRIPT":    "examples/maze_dr_egt.py",
        "USE_ACCEL": False,
    },
    {
        "RUN_NAME":  "accel_eval80_seed0_lr{lr}_30000c",
        "SCRIPT":    "examples/maze_plr.py",
        "USE_ACCEL": True,
    },
]
# =========================

In [ ]:
# Check GPU
!nvidia-smi

# JAX (CUDA 12 wheels; no local CUDA toolkit needed)
!pip -q install --upgrade pip
!pip -q install "jax[cuda12]"

# Sanity check
import jax
print("JAX:", jax.__version__)
print("Devices:", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), "No GPU visible to JAX"


Sun Sep 21 18:11:53 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os, sys, pathlib, subprocess

# Fresh clone
!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

# Optionally pin branch/commit
if GIT_CHECKOUT:
    !git checkout {GIT_CHECKOUT}
if GIT_COMMIT:
    !git checkout {GIT_COMMIT}

# Show current revision
!git --no-pager log -1 --pretty=format:"%h %ad %s" --date=iso

# Install package + example extras (no --no-deps here)
!pip -q install -e ".[examples]"

# Quick dep sanity
import flax, optax, chex, wandb, numpy as np
print("deps ok")


In [ ]:
# Ensure system metrics appear (CPU/GPU/RAM)
!pip -q install -U wandb psutil nvidia-ml-py3

import os, wandb
# Don’t disable stats/code
for k in ["WANDB_MODE", "WANDB_DISABLE_CODE", "WANDB_DISABLE_STATS"]:
    os.environ.pop(k, None)
# Help W&B agent connect; unbuffer training output
os.environ.setdefault("WANDB__SERVICE_WAIT", "300")
os.environ.setdefault("PYTHONUNBUFFERED", "1")

wandb.login()

# Mount Drive (for checkpoint persistence)
from google.colab import drive
from pathlib import Path

DRIVE_SAVE_DIR = None
if PERSIST_TO_DRIVE:
    drive.mount("/content/drive", force_remount=True)
    DRIVE_SAVE_DIR = Path("/content/drive/MyDrive") / DRIVE_DIR_NAME
    DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    print("Drive save dir:", DRIVE_SAVE_DIR)


  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'nvidia-ml-py3' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'nvidia-ml-py3'. Discussion can be found at https://github.com/pypa/pip/issues/6334


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: callumrlawson (callumrlawson-pibbss) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Mounted at /content/drive
Drive save dir: /content/drive/MyDrive/jaxued_checkpoints


In [ ]:
import sys, subprocess, pathlib, shutil, re
from pathlib import Path
from datetime import datetime

_SEED_REGEX = re.compile(r"seed\d+")

def _seeded_name(base:str, seed:int) -> str:
    """Replace 'seed<digits>' with the current seed; if absent, append '_seed<seed>'."""
    if _SEED_REGEX.search(base):
        return _SEED_REGEX.sub(f"seed{seed}", base)
    # fall back to appending a seed suffix
    return f"{base}_seed{seed}"

def run_one(script:str, run_name:str, seed:int, use_accel:bool, lr:float):
    """Launch a single training run (one seed) and stream logs."""
    script_path = Path(script)
    if not script_path.exists():
        raise FileNotFoundError(
            f"Expected training script not found: {script_path}\n"
            f"Check your repo/branch/commit, or fix SCRIPT."
        )

    # Build command (keep your existing settings)
    cmd = [
        sys.executable, "-u", str(script_path),      # -u = unbuffered
        "--project", PROJECT,
        "--run_name", run_name,
        "--seed", str(seed),
        "--num_updates", str(NUM_UPDATES),
        "--eval_freq", str(EVAL_FREQ),
        "--eval_num_attempts", str(EVAL_NUM),
        "--lr", str(lr),
    ] + (["--use_accel"] if use_accel else [])

    print("\n" + "="*80)
    print("Launching:\n ", " ".join(cmd))
    print("="*80)

    # Stream stdout/stderr so crash reason is visible
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"Training script exited with code {ret}")

def copy_checkpoints_to_drive(run_name:str, seed:int):
    """Copy checkpoints/<RUN_NAME>/<seed> to Drive; never overwrite (timestamp if needed)."""
    if not PERSIST_TO_DRIVE:
        return

    # Local source (exactly as training saved it)
    src = Path("checkpoints") / run_name / str(seed)

    # Drive base
    base = Path("/content/drive/MyDrive") / DRIVE_DIR_NAME / "checkpoints"

    # Default destination mirrors local structure:
    # <Drive>/<DRIVE_DIR_NAME>/checkpoints/<RUN_NAME>/<seed>/
    dst = base / run_name / str(seed)

    if not src.exists():
        print(f"❌ No checkpoints found at: {src}")
        return

    if dst.exists():
        ts = datetime.now().strftime("%Y%m%d-%H%M%S")
        print(f"ℹ Destination exists: {dst}")
        dst = base / f"{run_name}_{ts}" / str(seed)
        print(f"→ Saving this run to: {dst}")

    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)
    print("✅ Copied checkpoints")
    print("   src:", src)
    print("   dst:", dst)


In [ ]:
print("Starting learning rate grid search:")
print(f"Learning rates: {LEARNING_RATES}")
print(f"Seeds: {SEEDS}")
print(f"Algorithms: {[r['RUN_NAME'].split('_')[0] for r in RUNS]}")

for i, r in enumerate(RUNS, start=1):
    base_name  = r["RUN_NAME"]
    script     = r["SCRIPT"]
    use_accel  = bool(r.get("USE_ACCEL", False))

    print(f"\n### ALGORITHM {i}/{len(RUNS)}: {base_name.split('_')[0]} ###")

    for j, lr in enumerate(LEARNING_RATES):
        seed = SEEDS[j]  # One seed per LR

        # Convert LR to filename-safe format
        lr_mapping = {
            3e-5: "3e-5",
            7.5e-5: "75e-6",
            1.5e-4: "15e-5",
            2e-4: "2e-4",
            2.5e-4: "25e-5",
            4.5e-4: "45e-5",
            6e-4: "6e-4"
        }
        lr_str = lr_mapping[lr]

        # Replace {lr} placeholder with filename-safe learning rate
        base_name_with_lr = base_name.replace("{lr}", lr_str)

        # derive a neat per-seed name (replace seed0 → seed<seed>)
        run_name = _seeded_name(base_name_with_lr, seed)

        print(f"\n--- Learning Rate {lr} ({lr_str}) - Seed {seed} : {run_name} ---")
        run_one(script=script, run_name=run_name, seed=seed, use_accel=use_accel, lr=lr)
        copy_checkpoints_to_drive(run_name=run_name, seed=seed)

print("\n🎉 All learning rate grid search complete and checkpoints copied.")